# Phase 1: Data Extraction

## 1. Imports

In [1]:
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import time
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_GAMES_PATH = PROJECT_ROOT / 'data' / 'raw' / 'games_raw_24_26.csv'
CLEAN_GAMES_PATH = PROJECT_ROOT / 'data' / 'processed' / 'games_clean.csv'
REFRESH_API = False

## 2. Extract Game Data (2024-25, 2025-26 — Regular Season & Playoffs)

**Question:** What does our raw game-level dataset look like across the two 
training seasons, and does the row count match expectations?

We pull both season types separately per season, using `nba_api`'s live 
endpoint. Regular season pulls already include NBA Cup group stage and 
knockout games natively (confirmed via investigation — the NBA Cup 
Championship Final is the only game tagged separately, and it's excluded 
by default since it doesn't fall under 'Regular Season').

In [2]:
seasons = ['2024-25', '2025-26']
season_types = ['Regular Season', 'Playoffs']

if RAW_GAMES_PATH.exists() and not REFRESH_API:
    games_combined = pd.read_csv(RAW_GAMES_PATH, dtype={'GAME_ID': str})
    print(f'Loaded cached raw games from {RAW_GAMES_PATH.relative_to(PROJECT_ROOT)}')
else:
    all_games = []
    for season in seasons:
        for stype in season_types:
            gf = leaguegamefinder.LeagueGameFinder(
                season_nullable=season,
                season_type_nullable=stype
            )
            df = gf.get_data_frames()[0]
            df['SEASON_LABEL'] = season
            df['SEASON_TYPE_LABEL'] = stype
            all_games.append(df)
            time.sleep(1)  # To avoid hitting the API rate limit

    games_combined = pd.concat(all_games, ignore_index=True)
    RAW_GAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
    games_combined.to_csv(RAW_GAMES_PATH, index=False)
    print(f'Fetched from nba_api and cached to {RAW_GAMES_PATH.relative_to(PROJECT_ROOT)}')

Loaded cached raw games from data\raw\games_raw_24_26.csv


In [3]:
games_combined.groupby(['SEASON_LABEL', 'SEASON_TYPE_LABEL']).size()

SEASON_LABEL  SEASON_TYPE_LABEL
2024-25       Playoffs              168
              Regular Season       2460
2025-26       Playoffs              170
              Regular Season       2460
dtype: int64

**Takeaway:** Both the 2024-25 and 2025-26 regular seasons return the full 2,460 games, 
confirming both seasons are fully complete. Playoffs total 168 games in 
2024-25 and 170 in 2025-26 — the difference reflects series length 
variation (more games going to 6 or 7 games in 2025-26).


## 3. Sanity Check Before Saving

**Question:** Does anything look wrong before we lock this in as our 
clean dataset?

A quick check for nulls or unexpected duplicate rows protects the 
integrity of every feature built downstream in Phase 2.

In [4]:
# checking for null values across all columns
print("Null values:\n", games_combined.isnull().sum())

# checking for duplicate rows
print("\nDuplicate rows:", games_combined.duplicated().sum())

# confirm season type and season label columns are correct
print("\nUnique season types:", games_combined['SEASON_TYPE_LABEL'].unique())

Null values:
 SEASON_ID            0
TEAM_ID              0
TEAM_ABBREVIATION    0
TEAM_NAME            0
GAME_ID              0
GAME_DATE            0
MATCHUP              0
WL                   0
MIN                  0
PTS                  0
FGM                  0
FGA                  0
FG_PCT               0
FG3M                 0
FG3A                 0
FG3_PCT              0
FTM                  0
FTA                  0
FT_PCT               0
OREB                 0
DREB                 0
REB                  0
AST                  0
STL                  0
BLK                  0
TOV                  0
PF                   0
PLUS_MINUS           0
SEASON_LABEL         0
SEASON_TYPE_LABEL    0
dtype: int64

Duplicate rows: 0

Unique season types: <StringArray>
['Regular Season', 'Playoffs']
Length: 2, dtype: str


**Takeaway:** The data cleaning verification confirms the dataset is structured correctly and ready for analysis:

*   **No Missing Data**: There are zero null values present across all columns in the dataset.
    
*   **No Duplicate Rows**: The row-level integrity is maintained with zero duplicate records found.
    
*   **Consistent Categories**: The `SEASON_TYPE_LABEL` column contains only the expected `Regular Season` and `Playoffs` values, with no anomalies or typos.

## 4. Save Cleaned Dataset

Exporting the verified game-level dataset to `data/processed/` as a 
clean, versioned handoff to Phase 2 (feature engineering).

In [5]:
# save the combined dataframe to a CSV file to data/processed/
CLEAN_GAMES_PATH.parent.mkdir(parents=True, exist_ok=True)
games_combined.to_csv(CLEAN_GAMES_PATH, index=False)